# Silver Layer

## Objective

The Silver Layer transforms raw data into clean, validated, and enriched datasets.

This notebook performs:

- Duplicate removal
- Missing value handling
- Data validation
- Derived column creation
- Joining datasets
- Preparing data for Gold Layer

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
bronze_learners = spark.table("lms_bronze.bronze_learners")
bronze_courses = spark.table("lms_bronze.bronze_courses")
bronze_enrolments = spark.table("lms_bronze.bronze_enrolments")

In [0]:
print("="*60)
print("BRONZE TABLE COUNTS")
print("="*60)

print("Learners :", bronze_learners.count())
print("Courses :", bronze_courses.count())
print("Enrolments :", bronze_enrolments.count())

BRONZE TABLE COUNTS
Learners : 500
Courses : 60
Enrolments : 2000


In [0]:
silver_enrolments = bronze_enrolments.dropDuplicates(["enrolment_id"])

print("Before :", bronze_enrolments.count())
print("After  :", silver_enrolments.count())

Before : 2000
After  : 1990


In [0]:

silver_learners = bronze_learners.dropDuplicates(["learner_id"])
silver_courses = bronze_courses.dropDuplicates(["course_id"])
silver_enrolments = bronze_enrolments.dropDuplicates(["enrolment_id"])

In [0]:
print("Learners :", silver_learners.count())

print("Courses :", silver_courses.count())

print("Enrolments :", silver_enrolments.count())

Learners : 500
Courses : 60
Enrolments : 1990


In [0]:
from pyspark.sql.functions import col

bronze_courses.filter(
    col("instructor_name").isNull()
).count()

6

In [0]:
silver_courses = bronze_courses.fillna(
    {
        "instructor_name": "Unknown Instructor"
    }
)

In [0]:
silver_enrolments = silver_enrolments.fillna(
    {
        "feedback_rating": 0
    }
)

In [0]:
silver_enrolments = silver_enrolments.fillna(
    {
        "assessment_score": 0
    }
)

In [0]:
silver_enrolments = silver_enrolments.fillna(
    {
        "last_activity_date": "1900-01-01"
    }
)

In [0]:
display(
silver_enrolments.select(
    [
        count(
            when(
                col(c).isNull(),
                c
            )
        ).alias(c)

        for c in silver_enrolments.columns
    ]
)
)

enrolment_id,learner_id,course_id,enrol_date,expected_completion_date,actual_completion_date,status,progress_pct,last_activity_date,assessment_score,attempts,feedback_rating,certificate_issued,learning_duration_days
0,0,0,0,0,1172,0,0,329,0,0,0,0,1172


In [0]:
silver_enrolments = silver_enrolments.withColumn(
    "learning_duration_days",
    datediff(
        col("actual_completion_date"),
        col("enrol_date")
    )
)

In [0]:
silver_enrolments = silver_enrolments.withColumn(
    "completion_delay_days",
    datediff(
        col("actual_completion_date"),
        col("expected_completion_date")
    )
)

In [0]:
silver_enrolments = silver_enrolments.withColumn(
    "re_enrolled",

    when(
        col("attempts")>=2,
        "Yes"
    ).otherwise("No")
)

In [0]:
silver_enrolments = silver_enrolments.withColumn(

    "certificate_flag",

    when(
        col("certificate_issued")=="Yes",
        1
    ).otherwise(0)

)

In [0]:
silver_df = (

    silver_enrolments

    .join(
        silver_learners,
        "learner_id",
        "left"
    )

    .join(
        silver_courses,
        "course_id",
        "left"
    )

)

In [0]:
print("="*60)

print("FINAL DATASET")

print("="*60)

print("Rows :", silver_df.count())

print("Columns :", len(silver_df.columns))

display(silver_df)

FINAL DATASET
Rows : 1990
Columns : 30


course_id,learner_id,enrolment_id,enrol_date,expected_completion_date,actual_completion_date,status,progress_pct,last_activity_date,assessment_score,attempts,feedback_rating,certificate_issued,learning_duration_days,completion_delay_days,re_enrolled,certificate_flag,learner_name,email,phone_number,city,registration_date,subscription_type,course_title,category,instructor_id,instructor_name,duration_hours,difficulty_level,price_inr
CRS001,LRN0376,ENR00460,2024-01-13,2024-02-12,2024-02-21,Completed,100,2024-02-21,79.22,1,2,Yes,39,9,No,1,Preeti Bose,preeti.bose.376@hotmail.com,9573065240,Chandigarh,2023-03-25,Basic,Python for Data Science,Data Science,INS009,Suresh Gupta,15,Intermediate,2986
CRS012,LRN0078,ENR01153,2024-02-06,2024-03-17,null,In Progress,15,2024-03-27,0.0,1,0,No,null,null,No,0,Shreya Mehta,shreya.mehta.78@outlook.com,9671369594,Mumbai,2023-11-06,Basic,Motion Graphics & Animation,Design,INS013,Manish Verma,20,Beginner,201
CRS006,LRN0360,ENR00358,2024-01-31,2024-02-10,2024-02-03,Completed,100,2024-02-03,99.66,1,5,Yes,3,-7,No,1,Ritika Pillai,ritika.pillai.360@yahoo.com,9647598762,Bhopal,2022-01-24,Free,HTML & CSS Mastery,Web Development,INS015,Unknown Instructor,5,Intermediate,2276
CRS015,LRN0300,ENR01913,2024-01-23,2024-02-02,null,Dropped,12,2024-01-28,0.0,1,0,No,null,null,No,0,Megha Reddy,megha.reddy.300@outlook.com,9851027261,Lucknow,2022-07-20,Free,Azure Data Engineering,Cloud Computing,INS013,Manish Verma,5,Beginner,942
CRS036,LRN0193,ENR00921,2024-01-04,2024-01-24,null,In Progress,51,2024-03-26,0.0,1,0,No,null,null,No,0,Rahul Patel,rahul.patel.193@hotmail.com,9444721264,Chennai,2023-05-11,Premium,Advanced Python OOP,Data Science,INS006,Priya Singh,10,Advanced,7011
CRS006,LRN0320,ENR01282,2024-01-08,2024-01-18,2024-01-11,Completed,100,2024-01-11,93.7,1,4,Yes,3,-7,No,1,Preeti Chatterjee,preeti.chatterjee.320@rediffmail.com,9821009968,Delhi,2022-11-09,Free,HTML & CSS Mastery,Web Development,INS015,Unknown Instructor,5,Intermediate,2276
CRS007,LRN0207,ENR01309,2024-01-17,2024-02-26,null,Not Started,0,null,0.0,1,0,No,null,null,No,0,Amit Kumar,amit.kumar.207@rediffmail.com,9341252866,Bengaluru,2023-08-12,Basic,Business Analytics Essentials,Business,INS003,Arjun Patel,20,Advanced,10604
CRS052,LRN0104,ENR00947,2024-02-04,2024-04-24,2024-03-04,Completed,100,2024-03-04,97.98,1,4,Yes,29,-51,No,1,Pooja Agarwal,pooja.agarwal.104@yahoo.com,9968164535,Delhi,2023-06-07,Basic,Svelte & SvelteKit,Web Development,INS005,Deepak Joshi,40,Intermediate,4129
CRS018,LRN0393,ENR00254,2024-01-29,2024-02-28,null,Dropped,30,2024-02-08,0.0,3,0,No,null,null,Yes,0,Preeti Pillai,preeti.pillai.393@gmail.com,9833146591,Surat,2023-02-28,Basic,Flutter App Development,Mobile Development,INS006,Priya Singh,15,Beginner,875
CRS040,LRN0301,ENR00965,2024-02-14,2024-03-15,null,Not Started,0,null,0.0,1,0,No,null,null,No,0,Sunita Mehta,sunita.mehta.301@hotmail.com,9735160917,Surat,2023-08-27,Enterprise,TypeScript Advanced,Web Development,INS009,Suresh Gupta,15,Beginner,786


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS lms_silver;

In [0]:
silver_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("lms_silver.silver_lms")

In [0]:
%sql

SHOW TABLES IN lms_silver;

database,tableName,isTemporary
lms_silver,silver_lms,false


In [0]:
%sql

SELECT *

FROM lms_silver.silver_lms

LIMIT 20;

course_id,learner_id,enrolment_id,enrol_date,expected_completion_date,actual_completion_date,status,progress_pct,last_activity_date,assessment_score,attempts,feedback_rating,certificate_issued,learning_duration_days,completion_delay_days,re_enrolled,certificate_flag,learner_name,email,phone_number,city,registration_date,subscription_type,course_title,category,instructor_id,instructor_name,duration_hours,difficulty_level,price_inr
CRS001,LRN0376,ENR00460,2024-01-13,2024-02-12,2024-02-21,Completed,100,2024-02-21,79.22,1,2,Yes,39,9,No,1,Preeti Bose,preeti.bose.376@hotmail.com,9573065240,Chandigarh,2023-03-25,Basic,Python for Data Science,Data Science,INS009,Suresh Gupta,15,Intermediate,2986
CRS012,LRN0078,ENR01153,2024-02-06,2024-03-17,null,In Progress,15,2024-03-27,0.0,1,0,No,null,null,No,0,Shreya Mehta,shreya.mehta.78@outlook.com,9671369594,Mumbai,2023-11-06,Basic,Motion Graphics & Animation,Design,INS013,Manish Verma,20,Beginner,201
CRS006,LRN0360,ENR00358,2024-01-31,2024-02-10,2024-02-03,Completed,100,2024-02-03,99.66,1,5,Yes,3,-7,No,1,Ritika Pillai,ritika.pillai.360@yahoo.com,9647598762,Bhopal,2022-01-24,Free,HTML & CSS Mastery,Web Development,INS015,Unknown Instructor,5,Intermediate,2276
CRS015,LRN0300,ENR01913,2024-01-23,2024-02-02,null,Dropped,12,2024-01-28,0.0,1,0,No,null,null,No,0,Megha Reddy,megha.reddy.300@outlook.com,9851027261,Lucknow,2022-07-20,Free,Azure Data Engineering,Cloud Computing,INS013,Manish Verma,5,Beginner,942
CRS036,LRN0193,ENR00921,2024-01-04,2024-01-24,null,In Progress,51,2024-03-26,0.0,1,0,No,null,null,No,0,Rahul Patel,rahul.patel.193@hotmail.com,9444721264,Chennai,2023-05-11,Premium,Advanced Python OOP,Data Science,INS006,Priya Singh,10,Advanced,7011
CRS006,LRN0320,ENR01282,2024-01-08,2024-01-18,2024-01-11,Completed,100,2024-01-11,93.7,1,4,Yes,3,-7,No,1,Preeti Chatterjee,preeti.chatterjee.320@rediffmail.com,9821009968,Delhi,2022-11-09,Free,HTML & CSS Mastery,Web Development,INS015,Unknown Instructor,5,Intermediate,2276
CRS007,LRN0207,ENR01309,2024-01-17,2024-02-26,null,Not Started,0,null,0.0,1,0,No,null,null,No,0,Amit Kumar,amit.kumar.207@rediffmail.com,9341252866,Bengaluru,2023-08-12,Basic,Business Analytics Essentials,Business,INS003,Arjun Patel,20,Advanced,10604
CRS052,LRN0104,ENR00947,2024-02-04,2024-04-24,2024-03-04,Completed,100,2024-03-04,97.98,1,4,Yes,29,-51,No,1,Pooja Agarwal,pooja.agarwal.104@yahoo.com,9968164535,Delhi,2023-06-07,Basic,Svelte & SvelteKit,Web Development,INS005,Deepak Joshi,40,Intermediate,4129
CRS018,LRN0393,ENR00254,2024-01-29,2024-02-28,null,Dropped,30,2024-02-08,0.0,3,0,No,null,null,Yes,0,Preeti Pillai,preeti.pillai.393@gmail.com,9833146591,Surat,2023-02-28,Basic,Flutter App Development,Mobile Development,INS006,Priya Singh,15,Beginner,875
CRS040,LRN0301,ENR00965,2024-02-14,2024-03-15,null,Not Started,0,null,0.0,1,0,No,null,null,No,0,Sunita Mehta,sunita.mehta.301@hotmail.com,9735160917,Surat,2023-08-27,Enterprise,TypeScript Advanced,Web Development,INS009,Suresh Gupta,15,Beginner,786
